### Week 9 — Model Optimization

#### Task
Improve model performance using:
- Hyperparameter Tuning
- GridSearchCV

#### Deliverables

#### Model Performance Comparison

| Model | Before | After |
|---------|---------|---------|
| Model Name | Performance Score | Improved Score |

### Improvements
- Tuned model hyperparameters to achieve better performance.
- Used **GridSearchCV** to find the best parameter combination.
- Improved model accuracy and overall prediction quality.

## Skills Tested
- Model Optimization
- Hyperparameter Tuning
- GridSearchCV

### 1. Importing Libraries:


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Loading The Dataset:

In [47]:
df = pd.read_csv('tickets.csv')
df.head()

,text,category
0,Cannot login,Login issue
1,Unable to log in to my account,Login issue
2,Login page keeps failing,Login issue
3,Forgot password and cannot sign in,Login issue
4,Incorrect username or password error,Login issue


### 3.Text Preprocessing:

In [48]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize and lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token,pos='v') for token in tokens if token not in stopwords.words('english')]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(clean_text)
df[['text', 'cleaned_text', 'category']].head()

,text,cleaned_text,category
0,Cannot login,cannot login,Login issue
1,Unable to log in to my account,unable log account,Login issue
2,Login page keeps failing,login page keep fail,Login issue
3,Forgot password and cannot sign in,forget password cannot sign,Login issue
4,Incorrect username or password error,incorrect username password error,Login issue


### 4. TF-IDF Vectorization:

In [49]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['category']

print(f"Shape of feature matrix: {X.shape}")

Shape of feature matrix: (50, 238)


### 5. Train/Test Split:

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 40 samples
Test set size: 10 samples


### 6.Train The Model:

In [55]:
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('Logistic Regression Accuracy:', accuracy_score(y_test, lr_pred))
# print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.6


### 7.Model Optimization using GridSearchCV:

In [58]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameters to tune
param_grid = {
    'C': [0.1, 1, 10, 100],
    'solver': ['lbfgs', 'newton-cg'],           
    'max_iter': [100, 200]
}

# Create GridSearchCV
grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')

# Fit the model
grid_search.fit(X_train, y_train)

# Get best parameters and score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.2f}")

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
print(f"Test accuracy with tuned model: {accuracy_score(y_test, y_pred_tuned):.2f}")

Best parameters: {'C': 10, 'max_iter': 100, 'solver': 'lbfgs'}
Best cross-validation score: 0.75
Test accuracy with tuned model: 0.90


In [63]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params': {
            'C': [1, 10, 20],
            'kernel': ['rbf', 'linear']
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [1, 5, 10]
        }
    },
    'logistic_regression': {
        'model': LogisticRegression(solver='lbfgs'),
        'params': {
            'C': [1, 5, 10]
        }
    }
}

In [64]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(X_train, y_train)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.750,"{'C': 1, 'kernel': 'linear'}"
1,random_forest,0.675,{'n_estimators': 10}
2,logistic_regression,0.750,{'C': 5}
